# Revision computations — PSCV + ESM-2 autophagy paper

Run the cells in order. Nothing is retrained from scratch: the scripts reuse the
classifiers and predictions already saved in `pscv_work`.

**Before you start**, put the six `.py` files in one Google Drive folder, e.g.
`MyDrive/pscv_code/`:

```
pscv_common.py            (shared definitions — the others import it)
05_statistical_tests.py   DeLong + McNemar
06_ablation.py            feature-group ablation (Table 5)
07_shap_analysis.py       SHAP (Figures 17-19)
08_decision_boundary.py   UMAP decision landscapes (Figures 14-16)
09_external_benchmark.py  external benchmark (optional)
```

For a GPU: **Runtime → Change runtime type → T4 GPU**. Scripts 05 and 06 also work on CPU.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, glob

hits = glob.glob('/content/drive/MyDrive/**/pscv_common.py', recursive=True)

if hits:
    CODE_DIR = os.path.dirname(hits[0])
    os.chdir(CODE_DIR)
    print('Found the scripts in:', CODE_DIR)
    print('Files:', sorted(f for f in os.listdir(CODE_DIR) if f.endswith('.py')))
else:
    print('Not found under MyDrive. Check that the .py files were unzipped and uploaded,')
    print('or use the upload cell below instead.')
    print('\nFolders directly under MyDrive:')
    for d in sorted(os.listdir('/content/drive/MyDrive'))[:40]:
        print('  ', d)

Found the scripts in: /content/drive/MyDrive/pscv_code
Files: ['05_statistical_tests.py', '06_ablation.py', '07_shap_analysis.py', '08_decision_boundary.py', '09_external_benchmark.py', 'pscv_common.py']


## 2. Set the two folder paths

`CODE_DIR` = where you put the `.py` files.
`WORK_DIR` = your existing `pscv_work` folder (the one holding `results.pkl`,
`features.npz` and the `clf_*.pkl` checkpoints).

In [3]:
import os

CODE_DIR = '/content/drive/MyDrive/pscv_code'
WORK_DIR = '/content/drive/MyDrive/pscv_work'

os.chdir(CODE_DIR)          # scripts import pscv_common.py from here
print('code files :', sorted(f for f in os.listdir(CODE_DIR) if f.endswith('.py')))
print('work files :', sorted(os.listdir(WORK_DIR))[:12])

code files : ['05_statistical_tests.py', '06_ablation.py', '07_shap_analysis.py', '08_decision_boundary.py', '09_external_benchmark.py', 'pscv_common.py']
work files : ['clf_BayesianMLP_independent.pkl', 'clf_BlockMoE_independent.pkl', 'clf_CNN-BiLSTM_independent.pkl', 'clf_CNN_independent.pkl', 'clf_MLP_independent.pkl', 'clf_TabNet_independent.pkl', 'clf_TabResNet_independent.pkl', 'clf_Transformer_independent.pkl', 'clf_WideDeep_independent.pkl', 'clf_XGBoost_independent.pkl', 'features.npz', 'folds_10.json']


## 3. Install the packages Colab does not ship with

In [4]:
!pip install -q shap umap-learn
import torch, xgboost, sklearn, shap, umap
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())
print('xgboost', xgboost.__version__, '| sklearn', sklearn.__version__, '| shap', shap.__version__)

torch 2.11.0+cu130 | GPU: True
xgboost 3.4.1 | sklearn 1.6.1 | shap 0.52.0


## 4. DeLong + McNemar tests  (manuscript Section 3.1.2)

Runs in under a minute, CPU is fine. Writes to `WORK_DIR/stats/`.

In [5]:
WD = WORK_DIR
!python 05_statistical_tests.py --work_dir "{WD}"


AUC (DeLong 95% CI) and ACC (Wilson 95% CI)
       model    n    AUC  AUC_SE  AUC_lo  AUC_hi    ACC  ACC_lo  ACC_hi
   BlockMoE 4684 0.8980  0.0045  0.8892  0.9068 0.8177  0.8064  0.8285
    XGBoost 4684 0.8962  0.0045  0.8873  0.9050 0.8151  0.8037  0.8260
   WideDeep 4684 0.8953  0.0045  0.8864  0.9041 0.8106  0.7992  0.8216
        MLP 4684 0.8950  0.0046  0.8859  0.9040 0.8198  0.8085  0.8306
        CNN 4684 0.8929  0.0047  0.8838  0.9020 0.8177  0.8064  0.8285
BayesianMLP 4684 0.8893  0.0047  0.8801  0.8984 0.8076  0.7961  0.8187

Pairwise DeLong test
  model_A     model_B  AUC_A  AUC_B   dAUC      z      p  p_holm
BlockMoE     XGBoost 0.8980 0.8962 0.0018 0.6094 0.5422  1.0000
BlockMoE    WideDeep 0.8980 0.8953 0.0028 1.1011 0.2709  1.0000
BlockMoE         MLP 0.8980 0.8950 0.0031 1.2213 0.2220  1.0000
BlockMoE         CNN 0.8980 0.8929 0.0051 1.6391 0.1012  1.0000
BlockMoE BayesianMLP 0.8980 0.8893 0.0087 3.1455 0.0017  0.0249
 XGBoost    WideDeep 0.8962 0.8953 0.0009 0.3050 0

## 5. Feature-group ablation  (Table 5)

Retrains each classifier on PSCV only / ESM-2 only / fused, five seeds each.
About 25-40 minutes on a T4.

**It is resumable**: every finished run is saved to `WORK_DIR/ablation/ablation_preds.pkl`,
so if Colab disconnects, just run the cell again and it continues where it stopped.

In [6]:
WD = WORK_DIR
!python 06_ablation.py --work_dir "{WD}" --models BlockMoE XGBoost MLP CNN BayesianMLP

BlockMoE     PSCV  seed=42  AUC=0.7627 ACC=0.6904 MCC=0.3826  (31s)
BlockMoE     PSCV  seed=43  AUC=0.7441 ACC=0.6783 MCC=0.3562  (16s)
BlockMoE     PSCV  seed=44  AUC=0.7521 ACC=0.6772 MCC=0.3578  (32s)
BlockMoE     PSCV  seed=45  AUC=0.7518 ACC=0.6857 MCC=0.3712  (19s)
BlockMoE     PSCV  seed=46  AUC=0.7352 ACC=0.6742 MCC=0.3493  (12s)
BlockMoE     ESM2  seed=42  AUC=0.8995 ACC=0.8187 MCC=0.6379  (18s)
BlockMoE     ESM2  seed=43  AUC=0.8980 ACC=0.8183 MCC=0.6373  (13s)
BlockMoE     ESM2  seed=44  AUC=0.8994 ACC=0.8241 MCC=0.6480  (13s)
BlockMoE     ESM2  seed=45  AUC=0.8957 ACC=0.8126 MCC=0.6252  (11s)
BlockMoE     ESM2  seed=46  AUC=0.8931 ACC=0.8072 MCC=0.6142  (9s)
BlockMoE     FUSED seed=42  AUC=0.8991 ACC=0.8134 MCC=0.6268  (29s)
BlockMoE     FUSED seed=43  AUC=0.8971 ACC=0.8117 MCC=0.6246  (21s)
BlockMoE     FUSED seed=44  AUC=0.8944 ACC=0.8057 MCC=0.6119  (20s)
BlockMoE     FUSED seed=45  AUC=0.8982 ACC=0.8160 MCC=0.6319  (13s)
BlockMoE     FUSED seed=46  AUC=0.8958 ACC=0.8106

## 6. SHAP analysis  (Figures 17-19)

About 5-10 minutes on a T4. Writes figures to `WORK_DIR/plots/shap_v2/` and every
number quoted in the text to `WORK_DIR/shap_v2/shap_summary.json`.

In [7]:
WD = WORK_DIR
!python 07_shap_analysis.py --work_dir "{WD}" --n_background 100 --n_explain 300

/content/drive/MyDrive/pscv_code/pscv_common.py:356: UserWarning: [12:44:30] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  d = _Unpickler(f).load()
XGBoost: PSCV share of total |SHAP| = 2.1%  (best PSCV rank among 1433 features: 88)
MLP: PSCV share of total |SHAP| = 1.6%  (best PSCV rank among 1433 features: 223)
BlockMoE: PSCV share of total |SHAP| = 0.1%  (best PSCV rank among 1433 features: 1281)
saved -> /content/drive/MyDrive/pscv_work/plots/shap_v2 and /content/drive/MyDrive/pscv_work/shap_v2


## 7. Decision landscapes  (Figures 14-16)

The UMAP projection takes a few minutes the first time and is then cached in
`WORK_DIR/umap_projection.npy`, so re-runs are fast. Figures go to
`WORK_DIR/plots/boundary_v2/`.

In [8]:
WD = WORK_DIR
!python 08_decision_boundary.py --work_dir "{WD}"

I0000 00:00:1791549953.558239   10948 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
wrote /content/drive/MyDrive/pscv_work/plots/boundary_v2/boundary_BlockMoE.png
/content/drive/MyDrive/pscv_code/pscv_common.py:356: UserWarning: [12:46:37] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  d = _Unpickler(f).load()
wrote /content/drive/MyDrive/pscv_work/plots

## 8. Look at the results

In [9]:
WD = WORK_DIR
import pandas as pd, json, os
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')

print('=== AUC with DeLong 95% CI ===')
display(pd.read_csv(f'{WD}/stats/auc_ci_delong.csv'))

print('=== Pairwise DeLong ===')
display(pd.read_csv(f'{WD}/stats/pairwise_delong.csv'))

abl = f'{WD}/ablation/ablation_summary.csv'
if os.path.exists(abl):
    print('=== Ablation (Table 5 / Supplementary Table S4) ===')
    display(pd.read_csv(abl))

shp = f'{WD}/shap_v2/shap_summary.json'
if os.path.exists(shp):
    s = json.load(open(shp))
    for m, v in s['models'].items():
        print(m, '-> PSCV share of total |SHAP|:',
              f"{100*v['share_of_total_abs_shap']['PSCV (153)']:.2f}%",
              '| best PSCV rank of 1433:', v['best_rank_of_a_pscv_feature'])

=== AUC with DeLong 95% CI ===


,model,n,AUC,AUC_SE,AUC_lo,AUC_hi,ACC,ACC_lo,ACC_hi
0,BlockMoE,4684,0.8980,0.0045,0.8892,0.9068,0.8177,0.8064,0.8285
1,XGBoost,4684,0.8962,0.0045,0.8873,0.9050,0.8151,0.8037,0.8260
2,WideDeep,4684,0.8953,0.0045,0.8864,0.9041,0.8106,0.7992,0.8216
3,MLP,4684,0.8950,0.0046,0.8859,0.9040,0.8198,0.8085,0.8306
4,CNN,4684,0.8929,0.0047,0.8838,0.9020,0.8177,0.8064,0.8285
5,BayesianMLP,4684,0.8893,0.0047,0.8801,0.8984,0.8076,0.7961,0.8187


=== Pairwise DeLong ===


,model_A,model_B,AUC_A,AUC_B,dAUC,z,p,p_holm
0,BlockMoE,XGBoost,0.8980,0.8962,0.0018,0.6094,0.5422,1.0000
1,BlockMoE,WideDeep,0.8980,0.8953,0.0028,1.1011,0.2709,1.0000
2,BlockMoE,MLP,0.8980,0.8950,0.0031,1.2213,0.2220,1.0000
3,BlockMoE,CNN,0.8980,0.8929,0.0051,1.6391,0.1012,1.0000
4,BlockMoE,BayesianMLP,0.8980,0.8893,0.0087,3.1455,0.0017,0.0249
5,XGBoost,WideDeep,0.8962,0.8953,0.0009,0.3050,0.7603,1.0000
6,XGBoost,MLP,0.8962,0.8950,0.0012,0.3658,0.7145,1.0000
7,XGBoost,CNN,0.8962,0.8929,0.0033,1.1144,0.2651,1.0000
8,XGBoost,BayesianMLP,0.8962,0.8893,0.0069,2.0283,0.0425,0.5360
9,WideDeep,MLP,0.8953,0.8950,0.0003,0.1182,0.9059,1.0000


=== Ablation (Table 5 / Supplementary Table S4) ===


,model,features,AUC_mean,ACC_mean,MCC_mean,F1_mean,SENS_mean,SPEC_mean,AUC_sd,ACC_sd,MCC_sd,F1_sd,SENS_sd,SPEC_sd,n_runs
0,BayesianMLP,ESM2,0.8947,0.8105,0.6217,0.8051,0.8029,0.8178,0.0016,0.0030,0.0053,0.0057,0.0271,0.0274,5
1,BayesianMLP,FUSED,0.8972,0.8174,0.6346,0.8122,0.8094,0.8250,0.0023,0.0036,0.0071,0.0030,0.0052,0.0092,5
2,BayesianMLP,PSCV,0.7582,0.6864,0.3726,0.6761,0.6717,0.7004,0.0083,0.0047,0.0100,0.0106,0.0252,0.0189,5
3,BlockMoE,ESM2,0.8972,0.8162,0.6326,0.8114,0.8110,0.8211,0.0025,0.0058,0.0117,0.0071,0.0199,0.0181,5
4,BlockMoE,FUSED,0.8969,0.8115,0.6232,0.8086,0.8163,0.8069,0.0017,0.0034,0.0067,0.0040,0.0134,0.0136,5
5,BlockMoE,PSCV,0.7492,0.6812,0.3634,0.6631,0.6453,0.7154,0.0091,0.0060,0.0119,0.0154,0.0475,0.0502,5
6,CNN,ESM2,0.8938,0.8103,0.6204,0.8047,0.8011,0.8191,0.0017,0.0021,0.0041,0.0020,0.0091,0.0109,5
7,CNN,FUSED,0.8913,0.8086,0.6181,0.7993,0.7827,0.8333,0.0023,0.0034,0.0061,0.0085,0.0313,0.0274,5
8,CNN,PSCV,0.7560,0.6867,0.3741,0.6658,0.6414,0.7299,0.0028,0.0031,0.0054,0.0151,0.0405,0.0373,5
9,MLP,ESM2,0.8949,0.8122,0.6242,0.8067,0.8031,0.8209,0.0024,0.0039,0.0077,0.0038,0.0060,0.0073,5


XGBoost -> PSCV share of total |SHAP|: 2.12% | best PSCV rank of 1433: 88
MLP -> PSCV share of total |SHAP|: 1.61% | best PSCV rank of 1433: 223
BlockMoE -> PSCV share of total |SHAP|: 0.07% | best PSCV rank of 1433: 1281


In [10]:
WD = WORK_DIR
from IPython.display import Image, display
for f in ['plots/boundary_v2/boundary_BlockMoE.png',
          'plots/shap_v2/shap_pscv_BlockMoE.png',
          'plots/shap_v2/shap_blocks_all.png']:
    p = f'{WD}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(p, width=850))

Output hidden; open in https://colab.research.google.com to view.

## 9. External benchmark (optional — Section 3.4)

The reviewer asked for a head-to-head comparison with a published predictor. Upload the
positive and negative FASTA files of a published benchmark (e.g. the ATGPred-FL /
EnsembleDL-ATG independent test set) and run:

The script removes any external protein whose sequence also appears in your training set,
computes PSCV + ESM-2 features with the same code as notebook 02, and scores the saved
classifiers **without retraining**. Needs a GPU for the ESM-2 step.

In [12]:
WD = WORK_DIR
# !python 09_external_benchmark.py --work_dir "{WD}" \
#         --pos /content/ext_pos.fasta --neg /content/ext_neg.fasta --name ATGPredFL_test